# ETRI E2E Driving - 데이터 탐색 & 정합 검증

컨테이너 `etri-vad:cu128-nocompat` 안에서 실행합니다. 커널은 `/opt/venv/bin/python`.

**목적 두 가지**

1. 원본 데이터가 어떻게 생겼는지 파악
2. **3D 라벨 - 이미지 정합 검증.** 전처리(undistort / crop / scale) 이후 같은 그림을
   다시 그려 달라지지 않았는지 확인하는 기준선입니다. 정합이 깨지면 에러 없이
   성능만 떨어지므로 반드시 눈으로 확인해야 합니다.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

ROOT     = "/workspace/VAD/data/etri/train"
SCENARIO = sorted(os.listdir(ROOT))[0]
SDIR     = os.path.join(ROOT, SCENARIO)
FRAME    = 150          # 0 ~ 299
CAM      = "camera_front"

CAM_NAMES = ["camera_front", "camera_front_right", "camera_front_left",
             "camera_rear_wide", "camera_rear_left", "camera_rear_right"]

print("scenario :", SCENARIO)
print("frames   :", len(os.listdir(os.path.join(SDIR, "camera_front"))))

## 1. 캘리브레이션 - 카메라 6대

In [ ]:
calib = pd.read_parquet(SDIR + "/calibration/calibration.parquet").set_index("camera_name")

rows = []
for c in CAM_NAMES:
    r = calib.loc[c]
    K = np.array(r["K"], dtype=np.float64).reshape(3, 3)
    rows.append(dict(
        camera=c,
        fx=round(K[0, 0], 1), fy=round(K[1, 1], 1),
        cx=round(K[0, 2], 1), cy=round(K[1, 2], 1),
        fisheye=bool(r["is_fisheye"]),
        size="%dx%d" % (int(r["image_width"]), int(r["image_height"])),
        n_dist=len(np.array(r["distortion"])),
        t_xyz=np.round(np.array(r["translation"], dtype=float), 3).tolist(),
        euler_deg=np.round(np.array(r["euler"], dtype=float), 1).tolist()))

display(pd.DataFrame(rows).set_index("camera"))

`translation` 은 뒷바퀴 축 중심(원점) 기준 카메라 위치, `euler` 는 카메라에서 자차로의
회전(deg, xyz 순)입니다. `camera_rear_wide` 만 fisheye 이고 왜곡 계수 개수도 다릅니다.

## 2. 메타 - 타임스탬프 / 주행 명령

In [ ]:
ts  = pd.read_parquet(SDIR + "/meta/timestamps.parquet")
cmd = pd.read_parquet(SDIR + "/meta/command.parquet")

dt = np.diff(np.sort(ts["timestamp"].to_numpy()))
print("frame_id range : %d ~ %d   (n=%d)" % (ts.frame_id.min(), ts.frame_id.max(), len(ts)))
print("dt  mean=%.1f ms  std=%.2f  min=%d  max=%d" % (dt.mean(), dt.std(), dt.min(), dt.max()))
print()
print(cmd["command"].value_counts().to_string())

`frame_id` 가 -50 ~ 349 이면 과거 5초 + 메인 30초 + 미래 5초 구조입니다.
카메라 이미지는 0 ~ 299 에만 존재합니다.

## 3. 객체 라벨

In [ ]:
obj = pd.read_parquet(SDIR + "/annotation/object.parquet")
obj = obj.merge(ts[["timestamp", "frame_id"]], on="timestamp")
print("컬럼 :", list(obj.columns))
print()
print(obj["class"].value_counts().to_string())

ego_rows = obj[obj["class"] == "ego"]
dyn      = obj[obj["class"] != "ego"]
print()
print("ego 행 : %d  (프레임마다 1개여야 정상)" % len(ego_rows))
print("동적 객체 %d개,  고유 track ID %d개" % (len(dyn), dyn.obj_id.nunique()))

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
per_frame = dyn.groupby("frame_id").size()
ax[0].plot(per_frame.index, per_frame.values)
ax[0].set_title("objects per frame"); ax[0].set_xlabel("frame_id")
ax[1].hist(dyn.groupby("obj_id").size(), bins=40)
ax[1].set_title("track lifetime (frames)"); ax[1].set_yscale("log")
plt.tight_layout(); plt.show()

### 박스 치수 규약 - 일반적 관례와 반대입니다

| 컬럼 | 의미 |
|---|---|
| `width[m]`  | **진행방향(전후, x축)** 길이 |
| `length[m]` | **횡방향(좌우, y축)** 길이 |
| `height[m]` | 수직 높이 |
| `z[m]`      | **박스 바닥면** - 중심 높이 = `z + height/2` |

7절에서 이 규약이 실제로 맞는지 이미지 투영으로 확인합니다.

In [ ]:
d = dyn[dyn.frame_id == FRAME]
display(d.groupby("class")[["width[m]", "length[m]", "height[m]"]]
         .agg(["mean", "min", "max"]).round(2))

## 4. 자차 궤적

In [ ]:
pose = (pd.read_parquet(SDIR + "/annotation/ego_pose.parquet")
          .merge(ts[["timestamp", "frame_id"]], on="timestamp")
          .sort_values("frame_id"))

xy  = pose[["x", "y"]].to_numpy()
spd = np.linalg.norm(np.diff(xy, axis=0), axis=1) / (np.diff(pose.timestamp.to_numpy()) / 1e3)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(xy[:, 0], xy[:, 1], lw=1)
ax[0].scatter(*xy[0],  c="g", s=30, label="start")
ax[0].scatter(*xy[-1], c="r", s=30, label="end")
ax[0].axis("equal"); ax[0].set_title("ego trajectory (global)"); ax[0].legend()
ax[1].plot(pose.frame_id.iloc[1:], spd * 3.6); ax[1].set_title("speed (km/h)")
ax[2].plot(pose.frame_id, np.degrees(pose.yaw)); ax[2].set_title("yaw (deg)")
plt.tight_layout(); plt.show()

print("주행 거리 %.1f m, 평균 %.1f km/h" %
      (np.linalg.norm(np.diff(xy, axis=0), axis=1).sum(), spd.mean() * 3.6))

## 5. 좌표 변환 helper - 파이프라인과 동일한 수식

아래 함수들은 베이스라인 코드에서 그대로 옮겨온 것입니다.

- `undistorted_K` : `tools/data_converter/etri_vad_converter.py` 의 `undistorted_intrinsic()`
- `undistort`     : `pipelines/transform_3d.py` 의 `UndistortMultiViewImage`
- `lidar2img`     : `datasets/nuscenes_vad_dataset.py` 의 `get_data_info()`

In [ ]:
def euler_deg_to_R(e):
    return Rotation.from_euler("xyz", np.asarray(e, dtype=float), degrees=True).as_matrix()


def euler_rad_to_R(e):
    return Rotation.from_euler("xyz", np.asarray(e, dtype=float), degrees=False).as_matrix()


def undistorted_K(K, dist, size, is_fisheye):
    if is_fisheye:
        return cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
            K, dist[:4], size, np.eye(3), balance=0.0)
    newK, _ = cv2.getOptimalNewCameraMatrix(K, dist, size, alpha=0)
    return newK


def undistort(img, K, dist, newK, is_fisheye):
    h, w = img.shape[:2]
    if is_fisheye:
        m1, m2 = cv2.fisheye.initUndistortRectifyMap(
            K, dist[:4], np.eye(3), newK, (w, h), cv2.CV_32FC1)
    else:
        m1, m2 = cv2.initUndistortRectifyMap(
            K, dist, None, newK, (w, h), cv2.CV_32FC1)
    return cv2.remap(img, m1, m2, cv2.INTER_LINEAR)


def lidar2img(cam2ego_R, cam2ego_t, newK):
    r  = np.linalg.inv(cam2ego_R)
    t  = np.asarray(cam2ego_t, dtype=float) @ r.T
    rt = np.eye(4); rt[:3, :3] = r.T; rt[3, :3] = -t
    vp = np.eye(4); vp[:3, :3] = newK
    return vp @ rt.T


def project(pts, P):
    h   = np.concatenate([pts, np.ones((len(pts), 1))], axis=1)
    uvw = h @ P.T
    z   = uvw[:, 2]
    with np.errstate(divide="ignore", invalid="ignore"):
        uv = uvw[:, :2] / z[:, None]
    return uv, z


def global_to_ego_yaw(pts, ego_xyz, ego_yaw):
    # converter 의 박스 변환과 동일 (yaw 만 사용)
    c, s = np.cos(-ego_yaw), np.sin(-ego_yaw)
    R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    return (np.asarray(pts, dtype=float) - ego_xyz) @ R.T


def global_to_ego_full(pts, ego_xyz, rpy):
    # dataset 의 차선 변환과 동일 (roll/pitch/yaw 전체)
    return (np.asarray(pts, dtype=float) - ego_xyz) @ euler_rad_to_R(rpy)


EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
         (0, 4), (1, 5), (2, 6), (3, 7)]


def box_corners_global(r, swap_wl=False):
    lon, lat, h = r["width[m]"], r["length[m]"], r["height[m]"]   # 가이드 규약
    if swap_wl:
        lon, lat = lat, lon
    dx, dy = lon / 2.0, lat / 2.0
    c = np.array([[dx, dy, 0], [dx, -dy, 0], [-dx, -dy, 0], [-dx, dy, 0],
                  [dx, dy, h], [dx, -dy, h], [-dx, -dy, h], [-dx, dy, h]],
                 dtype=float)
    yaw = r["heading[rad]"]
    cs, sn = np.cos(yaw), np.sin(yaw)
    R = np.array([[cs, -sn, 0], [sn, cs, 0], [0, 0, 1]])
    return c @ R.T + np.array([r["x[m]"], r["y[m]"], r["z[m]"]], dtype=float)


print("helpers ready")

## 6. 차선 (map.parquet) - BEV

In [ ]:
lanes_raw = pd.read_parquet(SDIR + "/annotation/map.parquet")
lanes = [np.stack([np.asarray(p, dtype=float) for p in pts]) for pts in lanes_raw["points"]]
print("차선 %d개,  점 개수 min=%d max=%d" %
      (len(lanes), min(len(l) for l in lanes), max(len(l) for l in lanes)))

p = pose[pose.frame_id == FRAME].iloc[0]
e = ego_rows[ego_rows.frame_id == FRAME].iloc[0]
ego_xyz = np.array([p.x, p.y, p.z]); rpy = [p.roll, p.pitch, p.yaw]
box_xyz = np.array([e["x[m]"], e["y[m]"], e["z[m]"]]); box_yaw = e["heading[rad]"]

fig, ax = plt.subplots(figsize=(7, 7))
for l in lanes:
    q = global_to_ego_full(l[:, :3], ego_xyz, rpy)
    m = (np.abs(q[:, 0]) < 60) & (np.abs(q[:, 1]) < 40)
    if m.sum() > 1:
        ax.plot(q[m, 0], q[m, 1], lw=0.8, c="0.5")

col = {"Car": "tab:blue", "Pedestrian": "tab:red", "Cyclist": "tab:green"}
for _, r in dyn[dyn.frame_id == FRAME].iterrows():
    q = global_to_ego_yaw(box_corners_global(r)[:4], box_xyz, box_yaw)
    ax.fill(q[:, 0], q[:, 1], alpha=0.6, color=col.get(r["class"], "k"))

ax.plot(0, 0, "k*", ms=16); ax.arrow(0, 0, 4, 0, head_width=1, color="k")
ax.set_xlim(-20, 60); ax.set_ylim(-30, 30); ax.set_aspect("equal")
ax.set_xlabel("x forward (m)"); ax.set_ylabel("y left (m)")
ax.set_title("BEV  frame %d" % FRAME)
plt.show()

## 7. 이미지 투영 정합 검증 (핵심)

**전처리 후 이 그림이 동일하게 나와야 합니다.**
박스가 차량 위에 정확히 얹히고, 노란 차선이 노면 위에 붙어야 정상입니다.

In [ ]:
def render(cam_name, frame, swap_wl=False, draw_lanes=True, max_dist=60):
    r    = calib.loc[cam_name]
    K    = np.array(r["K"], dtype=float).reshape(3, 3)
    dist = np.array(r["distortion"], dtype=float)
    fish = bool(r["is_fisheye"])
    size = (int(r["image_width"]), int(r["image_height"]))
    newK = undistorted_K(K, dist, size, fish)
    P    = lidar2img(euler_deg_to_R(r["euler"]),
                     np.array(r["translation"], dtype=float), newK)

    img = cv2.imread("%s/%s/%08d.jpg" % (SDIR, cam_name, frame))
    img = cv2.cvtColor(undistort(img, K, dist, newK, fish), cv2.COLOR_BGR2RGB)

    pp = pose[pose.frame_id == frame].iloc[0]
    ee = ego_rows[ego_rows.frame_id == frame].iloc[0]
    exyz = np.array([pp.x, pp.y, pp.z]); rpy_ = [pp.roll, pp.pitch, pp.yaw]
    bxyz = np.array([ee["x[m]"], ee["y[m]"], ee["z[m]"]]); byaw = ee["heading[rad]"]

    if draw_lanes:
        for l in lanes:
            q = global_to_ego_full(l[:, :3], exyz, rpy_)
            q = q[(q[:, 0] > 0) & (np.linalg.norm(q[:, :2], axis=1) < max_dist)]
            if len(q) < 2:
                continue
            uv, z = project(q, P)
            ok = (z > 0.5) & (np.abs(uv[:, 0]) < 1e4) & (np.abs(uv[:, 1]) < 1e4)
            u2 = uv[ok].astype(int)
            for a, b in zip(u2[:-1], u2[1:]):
                cv2.line(img, tuple(a), tuple(b), (255, 220, 0), 3)

    bgr = {"Car": (60, 120, 255), "Pedestrian": (255, 60, 60), "Cyclist": (60, 220, 60)}
    for _, o in dyn[dyn.frame_id == frame].iterrows():
        c = global_to_ego_yaw(box_corners_global(o, swap_wl), bxyz, byaw)
        if np.linalg.norm(c[:, :2].mean(0)) > max_dist:
            continue
        uv, z = project(c, P)
        if (z <= 0.5).any() or np.abs(uv).max() > 1e4:
            continue
        uv = uv.astype(int)
        for a, b in EDGES:
            cv2.line(img, tuple(uv[a]), tuple(uv[b]), bgr.get(o["class"], (255, 255, 255)), 2)
    return img


plt.figure(figsize=(15, 12))
plt.imshow(render(CAM, FRAME)); plt.axis("off")
plt.title("%s  frame %d   undistorted + 3D box + lane" % (CAM, FRAME))
plt.show()

### 카메라 6대 전체

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 9))
for ax, c in zip(axes.ravel(), CAM_NAMES):
    ax.imshow(render(c, FRAME)); ax.set_title(c, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

### width / length 규약 확인

가이드 문서상 `width[m]` 가 진행방향입니다.
**왼쪽이 어긋나고 오른쪽이 맞다면** 규약이 반대이므로 컨버터의 `wlh` 순서를 다시 봐야 합니다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(19, 6))
ax[0].imshow(render(CAM, FRAME, swap_wl=False, draw_lanes=False))
ax[0].set_title("swap_wl=False   width=전후 (가이드 문서 기준)")
ax[1].imshow(render(CAM, FRAME, swap_wl=True, draw_lanes=False))
ax[1].set_title("swap_wl=True    반대")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

## 8. HD Map - 베이스라인 미사용, 심화 연구용

In [ ]:
hd  = pd.read_parquet(SDIR + "/annotation/hd_map.parquet")
hdp = (pd.read_parquet(SDIR + "/annotation/hd_ego_pose.parquet")
         .merge(ts[["timestamp", "frame_id"]], on="timestamp"))
print(hd["class"].value_counts().to_string())

o = hdp[hdp.frame_id == FRAME].iloc[0]
c, s = np.cos(-o.yaw), np.sin(-o.yaw)
R2 = np.array([[c, -s], [s, c]])

style = {"crosswalk": ("tab:purple", 1.5), "stop_line": ("tab:red", 2.0),
         "speed_bump": ("tab:orange", 1.5), "centerline": ("tab:green", 0.7)}

fig, ax = plt.subplots(figsize=(7, 7))
for _, el in hd.iterrows():
    q = (np.stack([np.asarray(p, dtype=float) for p in el["points"]]) - [o.x, o.y]) @ R2.T
    m = (np.abs(q[:, 0]) < 60) & (np.abs(q[:, 1]) < 40)
    if m.sum() < 2:
        continue
    cc, lw = style.get(el["class"], ("0.6", 0.8))
    ax.plot(q[m, 0], q[m, 1], lw=lw, color=cc)

ax.plot(0, 0, "k*", ms=16)
ax.set_xlim(-20, 60); ax.set_ylim(-30, 30); ax.set_aspect("equal")
ax.set_title("HD Map (2D, hd_ego_pose frame)")
plt.show()

---

## 전처리 후 재검증 방법

전처리(undistort -> crop -> scale)를 적용한 뒤에는 `render()` 에서

1. `undistort()` 호출 제거 (이미 적용된 이미지)
2. `newK` 를 **crop / scale 이 반영된 값**으로 교체
3. 이미지 경로를 전처리 디렉토리로 변경

만 바꾸고 같은 그림을 그립니다. **7절 그림이 전처리 전후로 동일해야 정상입니다.**